# Explicação das Técnicas Utilizadas
## Detecção de Área Queimada com Random Forest — Landsat 8 e 9

**Universidade Estadual Paulista "Júlio de Mesquita Filho" — UNESP ICT**  
**Disciplina:** Reconhecimento de Padrões — Prof. Dr. Rogério Galante Negri

---

Este notebook documenta **por que** cada técnica foi escolhida, **como** ela funciona matematicamente e **como** foi aplicada ao problema de detecção de área queimada a partir de imagens de satélite Landsat.

### Problema

Classificar cada pixel de uma imagem Landsat como **queimado (1)** ou **não queimado (0)** usando bandas espectrais e índices derivados de imagens pré e pós-evento de incêndio. Desafios principais:

- **Desbalanceamento severo**: ~3–5% queimado vs ~95–97% não queimado (~19:1)
- **Dependência espacial**: pixels vizinhos são fortemente correlacionados
- **Alta dimensionalidade local**: 12 features por pixel
- **NoData**: ~33% dos pixels sem informação válida

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    fbeta_score, matthews_corrcoef, cohen_kappa_score,
    accuracy_score, precision_recall_curve, average_precision_score
)
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler

np.random.seed(42)
print('Dependências carregadas.')

---
## 1. Por que Random Forest?

### 1.1 Motivação

O Random Forest foi escolhido como classificador por reunir características ideais para dados de sensoriamento remoto:

| Característica | Relevância para o problema |
|---|---|
| **Não paramétrico** | Não assume distribuição dos dados espectrais |
| **Robusto a outliers** | Pixels com ruído atmosférico ou de sensor não dominam o modelo |
| **Insensível à escala** | Bandas uint16 e índices float convivem sem normalização |
| **Importância de atributos** | Identifica quais índices (dNBR, NBR) mais discriminam área queimada |
| **OOB estimation** | Estima erro de generalização sem conjunto de validação separado |
| **Paralelizável** | Cada árvore é independente — treinamento eficiente |
| **Alta diversidade** | Bootstrap + atributos aleatórios reduzem variância sem aumentar viés |

### 1.2 Estrutura do Random Forest

O Random Forest combina $L$ árvores CART independentes:

```
Dataset D
    ├─ Bootstrap D₁ → CART₁ (X^(1) ⊆ X por nó)
    ├─ Bootstrap D₂ → CART₂ (X^(2) ⊆ X por nó)
    ├─ ...
    └─ Bootstrap DL → CARTL (X^(L) ⊆ X por nó)
                          ↓
               Majority Voting → g*(x)
```

O diferencial em relação ao Bagging: em cada nó de cada árvore, apenas $\sqrt{p}$ atributos são avaliados (sendo $p=12$ features). Isso **decorrelaciona** as árvores.

In [ ]:
# Demonstração: correlação entre árvores — Bagging vs Random Forest
# Dataset simulado com características similares ao problema (desbalanceado)
X_sim, y_sim = make_classification(
    n_samples=5000, n_features=12, n_informative=5, n_redundant=3,
    n_classes=2, weights=[0.95, 0.05],   # simula o desbalanceamento 19:1
    random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_sim, y_sim, test_size=0.3,
                                            stratify=y_sim, random_state=42)

# Compara diversidade: extrai predições de 10 árvores de cada ensemble
n_trees = 100
bag  = BaggingClassifier(estimator=DecisionTreeClassifier(),
                          n_estimators=n_trees, random_state=42).fit(X_tr, y_tr)
rf   = RandomForestClassifier(n_estimators=n_trees, random_state=42).fit(X_tr, y_tr)

# Correlação média entre predições das árvores
preds_bag = np.array([t.predict(X_te) for t in bag.estimators_])
preds_rf  = np.array([t.predict(X_te) for t in rf.estimators_])

corr_bag = np.corrcoef(preds_bag)
corr_rf  = np.corrcoef(preds_rf)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mat, titulo in [
    (axes[0], corr_bag, 'Bagging'),
    (axes[1], corr_rf,  'Random Forest')
]:
    im = ax.imshow(mat, cmap='hot_r', vmin=0, vmax=1)
    ax.set_title(f'{titulo}\nCorrelação média entre árvores = {np.triu(mat,1).mean():.3f}')
    ax.set_xlabel('Árvore')
    ax.set_ylabel('Árvore')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Correlação entre Predições das Árvores\n'
             '(menor correlação = mais diversidade = melhor ensemble)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Correlação média — Bagging     : {np.triu(corr_bag, 1).mean():.4f}')
print(f'Correlação média — Random Forest: {np.triu(corr_rf,  1).mean():.4f}')
print('→ RF produz árvores mais diversas, o que melhora o ensemble.')

---
## 2. CART — O Base Learner do Random Forest

### 2.1 O que é

Cada árvore do Random Forest é uma **CART** (*Classification and Regression Tree*). A CART divide recursivamente o espaço de atributos em regiões cada vez mais puras, usando limiares $\tau_{kh}$ sobre os atributos.

### 2.2 Critério de divisão — Entropia de Informação

A **impureza** de um nó $\mathcal{Q}$ é medida pela Entropia:

$$I(\mathcal{Q}) = -\sum_{j=1}^{c} P(\omega_j|\mathcal{Q}) \cdot \log_2 P(\omega_j|\mathcal{Q})$$

A divisão pelo limiar $\tau_{kh}$ é aceita quando maximiza a **redução de impureza**:

$$\Delta I(\mathcal{Q};\tau_{kh}) = I(\mathcal{Q}) - \frac{|\mathcal{Q}_{inf}|}{|\mathcal{Q}|}I(\mathcal{Q}_{inf}) - \frac{|\mathcal{Q}_{sup}|}{|\mathcal{Q}|}I(\mathcal{Q}_{sup})$$

Condições para aceitar a divisão:
- $\Delta I > \zeta$ (redução mínima de impureza)
- $|\mathcal{Q}| > \psi$ (mínimo de amostras no nó)

Se nenhuma divisão for aceita, o nó vira **folha** rotulada com a classe majoritária:
$$\omega^* = \arg\max_{\omega_j} P(\omega_j|\mathcal{Q})$$

### 2.3 Por que CART é ideal para o problema

- Índices espectrais como dNBR têm **limiares naturais** (ex: dNBR > 0.65 indica queima) — a CART encontra esses limiares automaticamente
- A divisão binária recursiva captura **interações não-lineares** entre bandas (ex: alta SWIR + baixo NIR = queimado)
- Não requer normalização: bandas em uint16 e índices em float[-1,1] convivem naturalmente

In [ ]:
# Visualiza como a CART aprende o limiar do dNBR
# Simula distribuições de dNBR para queimado e não queimado
np.random.seed(42)
n_nao   = 5000
n_quei  = 300
dnbr_nao   = np.random.normal(0.05, 0.15, n_nao)
dnbr_quei  = np.random.normal(0.75, 0.20, n_quei)
nbr_nao    = np.random.normal(0.40, 0.15, n_nao)
nbr_quei   = np.random.normal(-0.10, 0.20, n_quei)

X_demo = np.column_stack([
    np.concatenate([dnbr_nao, dnbr_quei]),
    np.concatenate([nbr_nao,  nbr_quei])
])
y_demo = np.array([0]*n_nao + [1]*n_quei)

cart_demo = DecisionTreeClassifier(max_depth=3, criterion='entropy', random_state=42)
cart_demo.fit(X_demo, y_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter com superfície de decisão
h = 0.01
xx, yy = np.meshgrid(np.arange(-0.5, 1.5, h), np.arange(-0.8, 1.0, h))
Z = cart_demo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlGn_r)
axes[0].scatter(dnbr_nao, nbr_nao, s=3, alpha=0.3, c='green', label='Não queimado')
axes[0].scatter(dnbr_quei, nbr_quei, s=10, alpha=0.6, c='red', label='Queimado')
axes[0].set_xlabel('dNBR')
axes[0].set_ylabel('NBR pré-evento')
axes[0].set_title('Superfície de Decisão CART (prof. 3)\nLimiares aprendidos automaticamente')
axes[0].legend(fontsize=9)
axes[0].axvline(0.65, color='black', linestyle='--', lw=1.5, label='Limiar convencional 0.65')
axes[0].legend(fontsize=8)

# Visualiza a estrutura da árvore
plot_tree(cart_demo, feature_names=['dNBR', 'NBR_pre'], class_names=['Não q.', 'Queimado'],
          filled=True, rounded=True, ax=axes[1], fontsize=8, impurity=True)
axes[1].set_title('Estrutura da Árvore CART\n(entropia, profundidade=3)')

plt.tight_layout()
plt.show()

# Mostra o limiar aprendido
threshold_dnbr = cart_demo.tree_.threshold[0]
print(f'Limiar aprendido pela CART no dNBR: {threshold_dnbr:.4f}')
print(f'Limiar convencional usado na máscara: 0.6500')
print('→ A CART encontra automaticamente o limiar mais discriminativo.')

---
## 3. Features — Por que Cada Atributo Foi Escolhido

### 3.1 Bandas Espectrais (B4, B5, B7)

| Banda | Comprimento de onda | Comportamento em área queimada |
|---|---|---|
| **B4 (Red)** | 0.64–0.67 µm | Aumenta levemente (carvão reflete no vermelho) |
| **B5 (NIR)** | 0.85–0.88 µm | **Cai drasticamente** (vegetação destruída) |
| **B7 (SWIR2)** | 2.11–2.29 µm | **Aumenta** (carvão e solo exposto absorvem menos SWIR) |

### 3.2 Índices Espectrais

**NDVI** (*Normalized Difference Vegetation Index*):
$$NDVI = \frac{NIR - Red}{NIR + Red} = \frac{B5 - B4}{B5 + B4} \in [-1, 1]$$

Mede vigor da vegetação. Área queimada: queda abrupta de NDVI pré→pós.

**NBR** (*Normalized Burn Ratio*):
$$NBR = \frac{NIR - SWIR2}{NIR + SWIR2} = \frac{B5 - B7}{B5 + B7} \in [-1, 1]$$

Projetado especificamente para detectar queimadas. Vegetação saudável: NBR alto. Área queimada: NBR muito baixo (até negativo).

**dNBR** e **dNDVI** (diferenças temporais):
$$dNBR = NBR_{pré} - NBR_{pós} \qquad dNDVI = NDVI_{pré} - NDVI_{pós}$$

Valores positivos indicam perda de vegetação. O limiar $dNBR > 0.65$ foi usado na máscara de referência.

### 3.3 Por que usar pré E pós?

Fornecer as 12 features ao RF (pré + pós + diferenças) permite que o modelo aprenda **contexto temporal completo**: uma área pode ter NBR baixo naturalmente (solo exposto, água) sem ter sido queimada. A combinação pré+pós+dNBR resolve essa ambiguidade.

In [ ]:
# Demonstra o poder discriminativo de cada feature
np.random.seed(42)
n_nao, n_que = 2000, 120

# Distribuições simuladas com base nos valores reais das imagens
features_sim = {
    'pre_B4'   : (np.random.normal(800,  150, n_nao),  np.random.normal(600,  200, n_que)),
    'pre_B5'   : (np.random.normal(2200, 400, n_nao),  np.random.normal(800,  300, n_que)),
    'pre_B7'   : (np.random.normal(1200, 200, n_nao),  np.random.normal(1800, 400, n_que)),
    'pre_NDVI' : (np.random.normal(0.45, 0.15, n_nao), np.random.normal(0.25, 0.15, n_que)),
    'pre_NBR'  : (np.random.normal(0.35, 0.15, n_nao), np.random.normal(0.10, 0.20, n_que)),
    'pos_B5'   : (np.random.normal(2100, 400, n_nao),  np.random.normal(600,  250, n_que)),
    'pos_NBR'  : (np.random.normal(0.33, 0.15, n_nao), np.random.normal(-0.35, 0.25, n_que)),
    'dNBR'     : (np.random.normal(0.02, 0.10, n_nao), np.random.normal(0.75,  0.20, n_que)),
    'dNDVI'    : (np.random.normal(0.01, 0.08, n_nao), np.random.normal(0.45,  0.15, n_que)),
}

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.ravel()

for ax, (nome, (v0, v1)) in zip(axes, features_sim.items()):
    # Sobreposição das distribuições (separabilidade visual)
    bins = np.linspace(min(v0.min(), v1.min()), max(v0.max(), v1.max()), 60)
    ax.hist(v0, bins=bins, alpha=0.6, color='#2ecc71', density=True, label='Não queimado')
    ax.hist(v1, bins=bins, alpha=0.7, color='#e74c3c', density=True, label='Queimado')

    # Índice de separabilidade (distância de Bhattacharyya simplificada)
    sep = abs(v0.mean() - v1.mean()) / (np.sqrt((v0.std()**2 + v1.std()**2) / 2) + 1e-6)
    ax.set_title(f'{nome}\nSeparabilidade: {sep:.2f}', fontsize=9)
    ax.set_xlabel('Valor', fontsize=8)
    ax.tick_params(labelsize=7)
    if nome == 'pre_B4':
        ax.legend(fontsize=7)

plt.suptitle('Poder Discriminativo das Features\n'
             '(maior separabilidade = feature mais informativa)', fontsize=12)
plt.tight_layout()
plt.show()

print('Separabilidade por feature (maior = melhor):')
seps = {}
for nome, (v0, v1) in features_sim.items():
    sep = abs(v0.mean() - v1.mean()) / (np.sqrt((v0.std()**2 + v1.std()**2) / 2) + 1e-6)
    seps[nome] = sep
for nome, sep in sorted(seps.items(), key=lambda x: -x[1]):
    barra = '█' * int(sep * 5)
    print(f'  {nome:<12}: {sep:5.2f} {barra}')

---
## 4. NoData — Por que Mascarar Antes de Tudo

### 4.1 O problema

As imagens Landsat possuem dois tipos de pixel inválido:

| Tipo | Valor sentinela | Origem |
|---|---|---|
| Bandas uint16 | `0` | Fora da faixa imageada ou sombra de nuvem |
| Índices float | `-3.4028e+38` (min float32) | Propagação de nodata na fórmula |

### 4.2 Por que não imputar

Imputação de valores espectrais em pixels de satélite é conceitualmente errada:
- NoData de satélite não é "dado faltante aleatório" — é ausência real de sinal físico
- Imputar criaria exemplos fictícios com assinaturas espectrais inválidas
- O modelo aprenderia padrões de pixels inventados

### 4.3 Solução aplicada

Máscara booleana `valida[i,j] = True` somente quando **todos** os 12 atributos e o rótulo têm valor válido. Apenas os pixels válidos entram na matriz de features $X$.

In [ ]:
# Demonstra o impacto de incluir NoData no treino
np.random.seed(42)
n = 3000
X_ok = np.random.randn(n, 3)
y_ok = (X_ok[:, 0] + X_ok[:, 1] > 0).astype(int)

# Adiciona pixels de NoData com valor -9999 e rótulo aleatório
n_nd = 500
X_nd = np.full((n_nd, 3), -9999.0)
X_nd += np.random.randn(n_nd, 3) * 0.01   # pequeno ruído
y_nd = np.random.randint(0, 2, n_nd)

X_com_nd = np.vstack([X_ok, X_nd])
y_com_nd = np.concatenate([y_ok, y_nd])

X_tr_ok,  X_te, y_tr_ok,  y_te = train_test_split(X_ok,     y_ok,     test_size=0.3, random_state=42)
X_tr_nd,  _,    y_tr_nd,  _    = train_test_split(X_com_nd, y_com_nd, test_size=0.3, random_state=42)

rf_ok = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_ok, y_tr_ok)
rf_nd = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_nd, y_tr_nd)

acc_ok = accuracy_score(y_te, rf_ok.predict(X_te))
acc_nd = accuracy_score(y_te, rf_nd.predict(X_te))

print('Impacto de incluir pixels NoData no treinamento:')
print(f'  RF treinado SEM NoData: acurácia = {acc_ok:.4f}')
print(f'  RF treinado COM NoData: acurácia = {acc_nd:.4f}')
print(f'  Degradação             : {acc_ok - acc_nd:.4f}')
print('→ Pixels NoData poluem o espaço de features e degradam o classificador.')

---
## 5. Desbalanceamento de Classes — O Problema Central

### 5.1 Por que é um problema

Com ~3–5% de pixels queimados, um classificador que sempre diz "não queimado" acerta **95–97%** dos casos. A acurácia global é, portanto, uma métrica enganosa.

O desbalanceamento faz o RF:
- **Subestimar a classe minoritária** — a função de custo é dominada pelos exemplos da maioria
- Criar árvores que raramente dividem pelo critério da classe queimada
- Produzir probabilidades muito baixas para `classe=1`, mesmo em pixels claramente queimados

### 5.2 Estratégias avaliadas

#### A) `class_weight='balanced'`
Ajusta os pesos das amostras inversamente proporcional à frequência da classe:
$$w_j = \frac{N}{c \cdot N_j}$$
onde $N$ = total de amostras, $c$ = número de classes, $N_j$ = amostras da classe $j$.

**Por que usar:** sem custo computacional adicional, sem alterar os dados. Cada erro na classe queimada pesa ~19x mais que na não queimada durante o crescimento da árvore.

#### B) Undersampling (RandomUnderSampler)
Remove aleatoriamente amostras da classe majoritária até atingir uma proporção desejada.

**Vantagem:** treino mais rápido.  
**Desvantagem:** descarta informação potencialmente útil da classe majoritária.

#### C) SMOTE (*Synthetic Minority Over-sampling Technique*)
Cria exemplos sintéticos da classe minoritária por **interpolação** entre vizinhos reais:

$$x_{novo} = x_i + \lambda \cdot (x_{vizinho} - x_i), \quad \lambda \in [0,1]$$

**Vantagem:** enriquece o espaço de features da classe minoritária com exemplos plausíveis.  
**Cuidado em dados espaciais:** SMOTE não considera posição geográfica — pode criar pixels "sintéticos" com combinações espectrais impossíveis.

#### D) SMOTETomek
Combina SMOTE (cria minoritários sintéticos) com **Tomek Links** (remove pares de amostras de classes diferentes muito próximas entre si — os mais ambíguos).

**Resultado:** fronteira de decisão mais limpa, sem amostras ambíguas.

In [ ]:
# Visualização do efeito de cada estratégia no espaço 2D
np.random.seed(42)
n_maj, n_min = 500, 30

X_maj = np.random.multivariate_normal([0, 0], [[1, 0.3],[0.3, 1]], n_maj)
X_min = np.random.multivariate_normal([2, 2], [[0.4, 0],[0, 0.4]], n_min)
X_2d  = np.vstack([X_maj, X_min])
y_2d  = np.array([0]*n_maj + [1]*n_min)

rus   = RandomUnderSampler(random_state=42)
smote = SMOTE(random_state=42, k_neighbors=min(5, n_min-1))
smt   = SMOTETomek(random_state=42)

X_under, y_under = rus.fit_resample(X_2d, y_2d)
X_smote, y_smote = smote.fit_resample(X_2d, y_2d)
X_smt,   y_smt   = smt.fit_resample(X_2d, y_2d)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

configs_plot = [
    (axes[0,0], X_2d,    y_2d,    f'Original\n({n_maj} não q. / {n_min} queimado)'),
    (axes[0,1], X_under, y_under, f'Undersampling\n({(y_under==0).sum()} não q. / {(y_under==1).sum()} queimado)'),
    (axes[1,0], X_smote, y_smote, f'SMOTE\n({(y_smote==0).sum()} não q. / {(y_smote==1).sum()} queimado)'),
    (axes[1,1], X_smt,   y_smt,   f'SMOTETomek\n({(y_smt==0).sum()} não q. / {(y_smt==1).sum()} queimado)'),
]

for ax, X_p, y_p, titulo in configs_plot:
    ax.scatter(X_p[y_p==0, 0], X_p[y_p==0, 1], c='#2ecc71', s=15, alpha=0.4, label='Não queimado')
    ax.scatter(X_p[y_p==1, 0], X_p[y_p==1, 1], c='#e74c3c', s=40, alpha=0.8, label='Queimado')
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlim(-4, 5)
    ax.set_ylim(-4, 5)
    ax.grid(alpha=0.2)

plt.suptitle('Efeito das Estratégias de Reamostagem no Espaço de Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Compara o impacto no classificador
np.random.seed(42)
X_bal, y_bal = make_classification(
    n_samples=5000, n_features=12, n_informative=5, n_redundant=3,
    n_classes=2, weights=[0.95, 0.05], random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_bal, y_bal, test_size=0.3,
                                            stratify=y_bal, random_state=42)

estrategias = {
    'Sem tratamento'       : (X_tr, y_tr, {}),
    'class_weight=balanced': (X_tr, y_tr, {'class_weight': 'balanced'}),
    'Undersampling'        : (*RandomUnderSampler(random_state=42).fit_resample(X_tr, y_tr), {}),
    'SMOTE'                : (*SMOTE(random_state=42).fit_resample(X_tr, y_tr), {}),
    'SMOTETomek'           : (*SMOTETomek(random_state=42).fit_resample(X_tr, y_tr), {}),
}

print(f"{'Estratégia':<25} {'OA':>7} {'F1':>7} {'F2':>7} {'MCC':>7} {'Recall':>8} {'Precision':>10}")
print('-' * 76)
resultados_bal = {}
for nome, (X_r, y_r, params) in estrategias.items():
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, **params)
    rf.fit(X_r, y_r)
    y_pred = rf.predict(X_te)
    metrics = {
        'OA': accuracy_score(y_te, y_pred),
        'F1': f1_score(y_te, y_pred),
        'F2': fbeta_score(y_te, y_pred, beta=2),
        'MCC': matthews_corrcoef(y_te, y_pred),
        'Recall': f1_score(y_te, y_pred, average=None)[1] if 1 in y_pred else 0,
        'Precision': f1_score(y_te, y_pred, average=None)[0],
    }
    from sklearn.metrics import recall_score, precision_score
    metrics['Recall']    = recall_score(y_te, y_pred, pos_label=1, zero_division=0)
    metrics['Precision'] = precision_score(y_te, y_pred, pos_label=1, zero_division=0)
    resultados_bal[nome] = metrics
    print(f"{nome:<25} {metrics['OA']:>7.4f} {metrics['F1']:>7.4f} {metrics['F2']:>7.4f} "
          f"{metrics['MCC']:>7.4f} {metrics['Recall']:>8.4f} {metrics['Precision']:>10.4f}")

---
## 6. Validação por Blocos Espaciais

### 6.1 O problema da divisão aleatória em dados espaciais

Pixels de imagem de satélite **não são independentes**. Um pixel e seu vizinho imediato compartilham:
- Condições atmosféricas idênticas
- Mesma cobertura vegetal de fundo
- Mesmo efeito de mistura espectral

Isso é chamado de **autocorrelação espacial**: pixels próximos são mais similares entre si do que pixels distantes.

Se dividirmos os pixels aleatoriamente em treino/teste, um pixel de treino e seu vizinho (de teste) são quase idênticos → o modelo "memoriza" a vizinhança → métricas **infladas artificialmente**.

### 6.2 Índice de Moran — medindo a autocorrelação

O Índice de Moran $I$ quantifica a autocorrelação espacial:
$$I = \frac{N}{\sum_{i}\sum_{j} w_{ij}} \cdot \frac{\sum_{i}\sum_{j} w_{ij}(x_i - \bar{x})(x_j - \bar{x})}{\sum_{i}(x_i - \bar{x})^2}$$

$I \approx 1$: alta autocorrelação (padrão clustering) → divisão aleatória é muito problemática.

### 6.3 Solução: Blocos Espaciais

Dividimos a imagem em um **grid $n \times n$** e alocamos blocos inteiros ao treino ou teste. Pixels de blocos distintos têm menor correlação espacial entre si.

In [ ]:
# Demonstra o problema da divisão aleatória vs blocos espaciais
np.random.seed(42)

# Simula uma imagem 100x100 com padrão espacial (autocorrelação alta)
from scipy.ndimage import gaussian_filter

ruido = np.random.randn(100, 100)
campo = gaussian_filter(ruido, sigma=10)   # suavização = autocorrelação
campo = (campo - campo.min()) / (campo.max() - campo.min())

# Rótulo: queimado onde campo > 0.65 (simula hotspot)
label_img = (campo > 0.65).astype(int)

# Feature: campo com ruído
feat_img = campo + np.random.randn(100, 100) * 0.05

# Flattens
X_img = feat_img.ravel().reshape(-1, 1)
y_img = label_img.ravel()
coords = np.argwhere(np.ones((100,100), bool))

# Divisão ALEATÓRIA
idx_al_tr = np.random.choice(len(y_img), int(0.7*len(y_img)), replace=False)
idx_al_te = np.setdiff1d(np.arange(len(y_img)), idx_al_tr)

# Divisão por BLOCOS (4x4 = 16 blocos, 4 para teste)
bloco_l = (coords[:, 0] // 25).clip(0, 3)
bloco_c = (coords[:, 1] // 25).clip(0, 3)
bloco_id = bloco_l * 4 + bloco_c
blocos_te = np.array([0, 5, 10, 15])   # diagonal
idx_bl_te = np.where(np.isin(bloco_id, blocos_te))[0]
idx_bl_tr = np.setdiff1d(np.arange(len(y_img)), idx_bl_te)

# Treina e avalia em ambos
resultados_split = {}
for nome, idx_tr, idx_te in [
    ('Divisão aleatória', idx_al_tr, idx_al_te),
    ('Blocos espaciais',  idx_bl_tr, idx_bl_te)
]:
    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X_img[idx_tr], y_img[idx_tr])
    y_p = rf.predict(X_img[idx_te])
    resultados_split[nome] = {
        'F1'  : f1_score(y_img[idx_te], y_p, zero_division=0),
        'F2'  : fbeta_score(y_img[idx_te], y_p, beta=2, zero_division=0),
        'OA'  : accuracy_score(y_img[idx_te], y_p),
    }

# Visualização
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(campo, cmap='YlGn')
axes[0].contour(label_img, levels=[0.5], colors='red', linewidths=2)
axes[0].set_title('Imagem simulada\n(vermelho = área queimada real)')
axes[0].axis('off')

# Mapa de divisão aleatória
mapa_al = np.zeros(10000)
mapa_al[idx_al_tr] = 0.3   # treino
mapa_al[idx_al_te] = 0.8   # teste
axes[1].imshow(mapa_al.reshape(100,100), cmap='RdBu', vmin=0, vmax=1)
axes[1].set_title(f'Divisão Aleatória\nF2={resultados_split["Divisão aleatória"]["F2"]:.4f} (inflado)')
axes[1].axis('off')

# Mapa de divisão por blocos
mapa_bl = np.zeros(10000)
mapa_bl[idx_bl_tr] = 0.3
mapa_bl[idx_bl_te] = 0.8
axes[2].imshow(mapa_bl.reshape(100,100), cmap='RdBu', vmin=0, vmax=1)
axes[2].set_title(f'Blocos Espaciais\nF2={resultados_split["Blocos espaciais"]["F2"]:.4f} (realista)')
axes[2].axis('off')

plt.suptitle('Divisão Aleatória vs Blocos Espaciais\n'
             '(azul=treino, vermelho=teste)', fontsize=12)
plt.tight_layout()
plt.show()

print('Comparação das métricas:')
for nome, met in resultados_split.items():
    print(f'  {nome:<22}: F2={met["F2"]:.4f}  OA={met["OA"]:.4f}')
print('→ Divisão aleatória infla artificialmente as métricas.')

---
## 7. Ajuste de Threshold via Curva Precision-Recall

### 7.1 Por que o threshold padrão (0.5) é inadequado

O RF retorna probabilidades $P(queimado|\mathbf{x}) \in [0,1]$. Por padrão, classifica como queimado quando $P > 0.5$.

Com classes desbalanceadas:
- A maioria dos pixels tem probabilidade muito baixa → threshold 0.5 é muito alto para a classe rara
- Resultado: **recall baixo** (muita área queimada não detectada)

### 7.2 Curva Precision-Recall

Varia o threshold $\tau$ de 0 a 1 e calcula para cada valor:
$$Precision(\tau) = \frac{TP(\tau)}{TP(\tau) + FP(\tau)} \qquad Recall(\tau) = \frac{TP(\tau)}{TP(\tau) + FN(\tau)}$$

**AUPRC** (*Area Under Precision-Recall Curve*): mais informativo que ROC-AUC com classes desbalanceadas, pois não considera TN — que são sempre abundantes no problema.

### 7.3 Threshold ótimo via F2

Para área queimada, **falsos negativos são mais custosos** que falsos positivos (área queimada não detectada é pior que alarme falso). Por isso priorizamos **recall** usando $F_\beta$ com $\beta = 2$:

$$F_2 = 5 \cdot \frac{Precision \cdot Recall}{4 \cdot Precision + Recall}$$

O threshold ótimo é:
$$\tau^* = \arg\max_{\tau} F_2(\tau)$$

In [ ]:
# Demonstra o ajuste de threshold
np.random.seed(42)
X_thr, y_thr = make_classification(
    n_samples=5000, n_features=8, n_informative=4,
    n_classes=2, weights=[0.95, 0.05], random_state=42
)
X_ttr, X_tte, y_ttr, y_tte = train_test_split(X_thr, y_thr, test_size=0.3,
                                               stratify=y_thr, random_state=42)
rf_thr = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                  random_state=42).fit(X_ttr, y_ttr)
y_prob_thr = rf_thr.predict_proba(X_tte)[:, 1]

precision_arr, recall_arr, thresholds = precision_recall_curve(y_tte, y_prob_thr)
thresholds_ext = np.append(thresholds, 1.0)

# F1 e F2 por threshold
denom_f1 = precision_arr + recall_arr
denom_f2 = 4 * precision_arr + recall_arr

f1_curve = np.where(denom_f1 > 0, 2 * precision_arr * recall_arr / denom_f1, 0)
f2_curve = np.where(denom_f2 > 0, 5 * precision_arr * recall_arr / denom_f2, 0)

thr_f1_opt = thresholds[np.argmax(f1_curve[:-1])]
thr_f2_opt = thresholds[np.argmax(f2_curve[:-1])]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Curva PR
axes[0].plot(recall_arr, precision_arr, 'b-', lw=2,
             label=f'AUPRC = {average_precision_score(y_tte, y_prob_thr):.3f}')
idx_05 = np.argmin(np.abs(thresholds - 0.5))
idx_f1 = np.argmax(f1_curve[:-1])
idx_f2 = np.argmax(f2_curve[:-1])
axes[0].scatter(recall_arr[idx_05], precision_arr[idx_05], c='gray',   s=120, zorder=5, label=f'τ=0.50')
axes[0].scatter(recall_arr[idx_f1], precision_arr[idx_f1], c='orange', s=120, zorder=5, label=f'τ*F1={thr_f1_opt:.2f}')
axes[0].scatter(recall_arr[idx_f2], precision_arr[idx_f2], c='red',    s=120, zorder=5, label=f'τ*F2={thr_f2_opt:.2f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Curva Precision-Recall')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# F1 e F2 por threshold
axes[1].plot(thresholds, f1_curve[:-1], 'orange', lw=2, label='F1-Score')
axes[1].plot(thresholds, f2_curve[:-1], 'red',    lw=2, label='F2-Score')
axes[1].axvline(0.5,        color='gray',   ls='--', label='τ=0.50')
axes[1].axvline(thr_f1_opt, color='orange', ls=':',  label=f'τ*F1={thr_f1_opt:.2f}')
axes[1].axvline(thr_f2_opt, color='red',    ls=':',  label=f'τ*F2={thr_f2_opt:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('F1/F2 por Threshold')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Comparação das métricas por threshold
for thr, cor, label in [
    (0.5, 'gray', 'τ=0.50'), (thr_f1_opt, 'orange', f'τ*F1={thr_f1_opt:.2f}'),
    (thr_f2_opt, 'red', f'τ*F2={thr_f2_opt:.2f}')
]:
    y_p = (y_prob_thr >= thr).astype(int)
    from sklearn.metrics import recall_score, precision_score
    metricas = [accuracy_score(y_tte,y_p), f1_score(y_tte,y_p,zero_division=0),
                fbeta_score(y_tte,y_p,beta=2,zero_division=0),
                recall_score(y_tte,y_p,pos_label=1,zero_division=0)]
    nomes = ['OA', 'F1', 'F2', 'Recall']
    axes[2].bar([n+' ' for n in nomes], metricas,
                alpha=0.7, color=cor, label=label, width=0.25,
                align='center')

axes[2].set_ylim(0, 1.1); axes[2].set_title('Métricas por Threshold')
axes[2].legend(fontsize=8); axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Ajuste de Threshold — Impacto nas Métricas', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Threshold padrão (0.50): F2={fbeta_score(y_tte,(y_prob_thr>=0.50).astype(int),beta=2,zero_division=0):.4f}')
print(f'Threshold ótimo F2 ({thr_f2_opt:.2f}): F2={fbeta_score(y_tte,(y_prob_thr>=thr_f2_opt).astype(int),beta=2,zero_division=0):.4f}')
print('→ Threshold menor captura mais área queimada (maior recall), com alguma perda de precision.')

---
## 8. Métricas de Avaliação — Por que Cada Uma Foi Escolhida

### 8.1 O problema da Acurácia Global (OA)

Com 95% de não queimado, um classificador trivial (sempre prediz 0) atinge **OA = 95%**. A OA é enganosa com classes desbalanceadas.

### 8.2 Métricas escolhidas

**F2-Score** ($\beta = 2$) — métrica principal:
$$F_2 = 5 \cdot \frac{Precision \cdot Recall}{4 \cdot Precision + Recall}$$
- Dá **peso 4x maior ao Recall** (detectar área queimada é mais importante que não gerar alarme falso)
- Um FN (queimada não detectada) é mais grave que um FP (alarme falso) no contexto ambiental

**MCC** (*Matthews Correlation Coefficient*):
$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$
- Leva em conta **todos os quadrantes** da matriz de confusão
- Robusto a desbalanceamento: $MCC \in [-1, 1]$, onde 0 = predição aleatória
- Preferível ao F-score quando TN importa (e importa: confirmar áreas não queimadas é útil para gestão)

**Coeficiente Kappa:**
$$\kappa = \frac{OA - OA_{esperada}}{1 - OA_{esperada}}$$
- Desconta acertos aleatórios da OA
- Padrão em avaliação de mapas temáticos de sensoriamento remoto

**AUPRC** (*Area Under Precision-Recall Curve*):
- Resumo escalar da curva PR em todos os thresholds
- Mais informativo que ROC-AUC com desbalanceamento (ROC é otimista por incluir TN)

**Precision e Recall separados:**
- Monitorar individualmente para entender o trade-off ao variar o threshold

In [ ]:
# Demonstra por que OA é enganosa
print('=== Problema da Acurácia Global com Desbalanceamento ===')
print()

n_total = 10000
n_quei  = 300
n_nao   = n_total - n_quei
y_real  = np.array([1]*n_quei + [0]*n_nao)

# Classificador trivial: sempre prediz 0
y_trivial = np.zeros(n_total, dtype=int)

# Classificador com alguns acertos na classe queimada
y_bom = np.zeros(n_total, dtype=int)
y_bom[:int(n_quei*0.75)] = 1   # detecta 75% das queimadas
y_bom[n_quei:n_quei+50] = 1    # 50 falsos positivos

print(f"{'Classificador':<30} {'OA':>7} {'F2':>7} {'MCC':>7} {'Recall':>8}")
print('-' * 60)
for nome, y_p in [('Trivial (sempre 0)', y_trivial), ('Útil (75% recall)', y_bom)]:
    from sklearn.metrics import recall_score
    oa  = accuracy_score(y_real, y_p)
    f2  = fbeta_score(y_real, y_p, beta=2, zero_division=0)
    mcc = matthews_corrcoef(y_real, y_p)
    rec = recall_score(y_real, y_p, pos_label=1, zero_division=0)
    print(f"{nome:<30} {oa:>7.4f} {f2:>7.4f} {mcc:>7.4f} {rec:>8.4f}")

print()
print('→ Classificador trivial tem OA=97% mas F2=0 e MCC=0.')
print('→ OA sozinha é completamente inadequada para este problema.')

In [ ]:
# Visualiza o que cada métrica mede
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de confusão com anotações explicativas
cm_data = np.array([[TN := 9650, FP := 50], [FN := 75, TP := 225]])
ax = axes[0]
im = ax.imshow(cm_data, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred: Não queimado', 'Pred: Queimado'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['Real: Não queimado', 'Real: Queimado'])
ax.text(0, 0, f'TN = {TN}\n(acerto classe 0)', ha='center', va='center', fontsize=11, color='navy')
ax.text(1, 0, f'FP = {FP}\n(alarme falso)', ha='center', va='center', fontsize=11, color='darkred')
ax.text(0, 1, f'FN = {FN}\n(queimada não detectada)', ha='center', va='center', fontsize=11, color='darkred')
ax.text(1, 1, f'TP = {TP}\n(acerto classe 1)', ha='center', va='center', fontsize=11, color='darkgreen')
ax.set_title('Matriz de Confusão Anotada', fontsize=11)

# Barras das métricas
metricas = {
    'OA'       : (TN+TP)/(TN+FP+FN+TP),
    'Precision': TP/(TP+FP),
    'Recall'   : TP/(TP+FN),
    'F1'       : 2*TP/(2*TP+FP+FN),
    'F2'       : 5*TP/(5*TP+4*FN+FP),
    'Kappa'    : cohen_kappa_score([0]*9700+[1]*300,
                                    [0]*9650+[1]*50+[0]*75+[1]*225),
    'MCC'      : (TP*TN-FP*FN)/np.sqrt((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN)),
}
cores_met = ['#e74c3c','#3498db','#2ecc71','#f39c12','#e67e22','#9b59b6','#1abc9c']
axes[1].barh(list(metricas.keys()), list(metricas.values()), color=cores_met, alpha=0.85)
for i, (nome, val) in enumerate(metricas.items()):
    axes[1].text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=10)
axes[1].set_xlim(0, 1.15)
axes[1].set_xlabel('Valor')
axes[1].set_title('Métricas de Avaliação\n(mesmo classificador)', fontsize=11)
axes[1].axvline(1.0, color='gray', ls='--', lw=1)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('O que cada métrica mede — Desbalanceamento 97:3', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Importância de Atributos

### 9.1 Como o Random Forest calcula

Para cada atributo $k$, a importância é a **redução média de impureza** (entropia) que ele proporciona ao longo de todas as árvores e todos os nós onde foi selecionado:

$$Imp(k) = \frac{1}{L} \sum_{l=1}^{L} \sum_{t \in \text{nós de }l \text{ usando } k} p(t) \cdot \Delta I(t)$$

onde $p(t) = |\mathcal{Q}_t| / N$ é a proporção de amostras no nó $t$.

### 9.2 Por que é útil no contexto do problema

- Confirma se o **dNBR domina** a discriminação (como esperado pela física)
- Identifica features redundantes que podem ser removidas sem perda de desempenho
- Valida que bandas individuais (B4, B5, B7) contribuem além dos índices derivados
- Pode revelar se o modelo aprendeu relações físicas corretas ou artefatos

In [ ]:
# Demonstra importância de atributos no problema simulado
np.random.seed(42)
n_nao, n_que = 3000, 180

# Simula as 12 features com graus variados de relevância
feat_names = ['pre_b4','pre_b5','pre_b7','pre_ndvi','pre_nbr',
              'pos_b4','pos_b5','pos_b7','pos_ndvi','pos_nbr',
              'dnbr','dndvi']

X_sim_full = np.random.randn(n_nao + n_que, 12)
# dNBR (col 10) e NBR pós (col 9) têm maior poder discriminativo
X_sim_full[:n_nao, 10] = np.random.normal(0.02, 0.10, n_nao)   # não queimado
X_sim_full[n_nao:, 10] = np.random.normal(0.80, 0.18, n_que)   # queimado
X_sim_full[:n_nao,  9] = np.random.normal(0.35, 0.12, n_nao)
X_sim_full[n_nao:,  9] = np.random.normal(-0.25, 0.20, n_que)
X_sim_full[:n_nao,  4] = np.random.normal(0.40, 0.15, n_nao)   # pre_nbr
X_sim_full[n_nao:,  4] = np.random.normal(0.20, 0.18, n_que)
X_sim_full[:n_nao, 11] = np.random.normal(0.01, 0.08, n_nao)   # dndvi
X_sim_full[n_nao:, 11] = np.random.normal(0.42, 0.15, n_que)

y_sim_full = np.array([0]*n_nao + [1]*n_que)
X_sf_tr, X_sf_te, y_sf_tr, y_sf_te = train_test_split(
    X_sim_full, y_sim_full, test_size=0.3, stratify=y_sim_full, random_state=42
)
rf_imp = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=42, n_jobs=-1).fit(X_sf_tr, y_sf_tr)

importancias = rf_imp.feature_importances_
idx_ord = np.argsort(importancias)[::-1]

def cor_feature(nome):
    if nome.startswith('d'): return '#e74c3c'
    if 'pre' in nome:        return '#3498db'
    return '#f39c12'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Importância
cores_imp = [cor_feature(feat_names[i]) for i in idx_ord]
axes[0].bar(range(12), importancias[idx_ord], color=cores_imp, alpha=0.85)
axes[0].set_xticks(range(12))
axes[0].set_xticklabels([feat_names[i] for i in idx_ord], rotation=45, ha='right', fontsize=9)
axes[0].set_ylabel('Importância (redução média de impureza)')
axes[0].set_title('Importância de Atributos — RF')
axes[0].grid(axis='y', alpha=0.3)
patches_imp = [
    mpatches.Patch(color='#e74c3c', label='Diferença temporal (dNBR, dNDVI)'),
    mpatches.Patch(color='#3498db', label='Pré-evento'),
    mpatches.Patch(color='#f39c12', label='Pós-evento'),
]
axes[0].legend(handles=patches_imp, fontsize=9)

# Importância acumulada
imp_acum = np.cumsum(importancias[idx_ord])
axes[1].plot(range(1, 13), imp_acum, 'b-o', ms=6)
axes[1].axhline(0.90, color='red', ls='--', label='90% da importância')
axes[1].axhline(0.95, color='orange', ls='--', label='95% da importância')
n_90 = np.argmax(imp_acum >= 0.90) + 1
n_95 = np.argmax(imp_acum >= 0.95) + 1
axes[1].axvline(n_90, color='red',    ls=':', label=f'{n_90} features → 90%')
axes[1].axvline(n_95, color='orange', ls=':', label=f'{n_95} features → 95%')
axes[1].set_xlabel('Número de features (por importância)')
axes[1].set_ylabel('Importância acumulada')
axes[1].set_title('Importância Acumulada')
axes[1].legend(fontsize=9)
axes[1].set_xticks(range(1, 13))
axes[1].grid(alpha=0.3)

plt.suptitle('Análise de Importância de Atributos', fontsize=13)
plt.tight_layout()
plt.show()

print('Interpretação:')
print(f'  → Apenas {n_90} features explicam 90% da discriminação')
print(f'  → Features de diferença temporal (dNBR, dNDVI) dominam — confirmando a física')
print(f'  → Features redundantes podem ser removidas sem perda significativa')

---
## 10. Parametrização — Randomized Search

### 10.1 Hiperparâmetros do Random Forest

| Parâmetro | Efeito no modelo | Faixa típica |
|---|---|---|
| `n_estimators` (L) | Mais árvores = menor variância, mas maior custo | 100–500 |
| `max_features` | Atributos por nó — controla diversidade | `sqrt(p)`, `log2(p)`, 0.5 |
| `min_samples_leaf` ($\psi$) | Folhas menores = mais flexibilidade | 1–20 |
| `max_depth` | Profundidade máxima = regularização | None, 10, 20, 30 |
| `min_impurity_decrease` ($\zeta$) | Redução mínima de entropia para dividir | 0, 1e-5, 1e-4 |
| `class_weight` | Ponderação das classes | `balanced`, `balanced_subsample` |

### 10.2 Por que Randomized Search em vez de Grid Search

Grid Search avalia **todas** as combinações: com 6 parâmetros e as faixas acima, seria $3 \times 3 \times 4 \times 4 \times 3 \times 2 = 864$ combinações, cada uma com 5 folds = **4.320 treinamentos**. Inviável computacionalmente.

**Randomized Search** amostra aleatoriamente $n_{iter}$ combinações de distribuições contínuas/discretas. Com $n_{iter} = 30$ (3.5% do espaço), a probabilidade de encontrar uma configuração no top-5% é:
$$P(\text{encontrar top-5\%}) = 1 - (1 - 0.05)^{30} \approx 0.785$$

79% de chance de achar uma configuração excelente com apenas 30 iterações.

In [ ]:
# Demonstra a diferença entre Grid Search e Randomized Search
from scipy.stats import randint as sp_randint

# Simula espaço de busca 2D para visualização
np.random.seed(42)

# Simula uma "superfície de desempenho" (F2 em função de 2 hiperparâmetros)
n_est_range   = np.linspace(50, 400, 100)
min_leaf_range = np.linspace(1, 20, 100)
N, M = np.meshgrid(n_est_range, min_leaf_range)

# Superfície fictícia com máximo em ~n_estimators=200, min_leaf=5
perf = (np.exp(-((N-200)**2)/(2*80**2)) *
        np.exp(-((M-5)**2)/(2*3**2)) * 0.4 + 0.5 +
        np.random.randn(*N.shape) * 0.02)

# Pontos do Grid Search (grade regular)
gs_n   = [100, 200, 300]
gs_ml  = [1, 5, 10, 15]
gs_pts = [(n, m) for n in gs_n for m in gs_ml]

# Pontos do Randomized Search (aleatórios)
rs_n   = np.random.uniform(50, 400, 20)
rs_ml  = np.random.uniform(1, 20, 20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, titulo, pts_x, pts_y in [
    (axes[0], 'Grid Search\n(12 pontos fixos)', [p[0] for p in gs_pts], [p[1] for p in gs_pts]),
    (axes[1], 'Randomized Search\n(20 pontos aleatórios)', rs_n, rs_ml)
]:
    im = ax.contourf(N, M, perf, levels=20, cmap='viridis')
    ax.scatter(pts_x, pts_y, c='white', s=80, zorder=5, edgecolors='red', lw=1.5, label='Pontos avaliados')
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('min_samples_leaf')
    ax.set_title(titulo)
    ax.legend(fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, label='F2 (simulado)')

plt.suptitle('Grid Search vs Randomized Search — Exploração do Espaço de Hiperparâmetros',
             fontsize=12)
plt.tight_layout()
plt.show()

# Probabilidade teórica de encontrar o top-k%
print('Probabilidade de encontrar configuração no top-X% (Randomized Search):')
for top_pct in [1, 5, 10]:
    for n_iter in [10, 20, 30, 50]:
        prob = 1 - (1 - top_pct/100)**n_iter
        print(f'  top-{top_pct:2d}%, n_iter={n_iter:3d}: P = {prob:.3f}')
    print()

---
## 11. Validação Cross-Cena — Generalização Espacial Real

### 11.1 Motivação

Treinar e testar na mesma cena (mesmo com blocos espaciais) ainda compartilha:
- Mesmas condições atmosféricas gerais
- Mesmo sensor (Landsat 8 ou 9)
- Mesma vegetação dominante da região

A validação **cross-cena** (treina em 221_074, testa em 220_075) é o teste mais severo: avalia se o modelo generaliza para uma cena diferente, com sensor diferente (L8 vs L9) e outra região geográfica.

### 11.2 O que uma queda de desempenho cross-cena indica

- O modelo aprendeu características específicas do sensor ou da vegetação local → necessidade de calibração
- Diferenças radiométricas entre Landsat 8 e 9 afetam os índices → normalização cross-sensor pode ser necessária
- A variabilidade espectral de "área queimada" é maior do que o modelo aprendeu

In [ ]:
# Demonstra o conceito de generalização cross-domínio
np.random.seed(42)

# Cena A (treino): distribuição 1
def gerar_cena(media_nao, media_que, seed, n_nao=2000, n_que=120):
    rng = np.random.default_rng(seed)
    X0 = rng.multivariate_normal(media_nao, np.eye(2)*0.04, n_nao)
    X1 = rng.multivariate_normal(media_que, np.eye(2)*0.06, n_que)
    return np.vstack([X0,X1]), np.array([0]*n_nao+[1]*n_que)

X_cA, y_cA = gerar_cena([0.10, 0.35], [0.75, -0.20], seed=1)
X_cB, y_cB = gerar_cena([0.08, 0.38], [0.70, -0.15], seed=2)  # cena B: distribuição similar mas deslocada
X_cC, y_cC = gerar_cena([0.12, 0.40], [0.65, -0.10], seed=3)  # cena C: mais diferente

rf_cross = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                    random_state=42).fit(X_cA, y_cA)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

h = 0.02
xx, yy = np.meshgrid(np.arange(-0.3, 1.2, h), np.arange(-0.6, 0.8, h))
Z = rf_cross.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

for ax, (X_c, y_c, nome) in zip(axes, [
    (X_cA, y_cA, 'Cena A (Treino / 221_074)'),
    (X_cB, y_cB, 'Cena B (Teste / 220_075)\n[similar]'),
    (X_cC, y_cC, 'Cena C (Hipotética)\n[mais diferente]'),
]):
    ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.5)
    ax.scatter(X_c[y_c==0,0], X_c[y_c==0,1], c='#2ecc71', s=5, alpha=0.4, label='Não queimado')
    ax.scatter(X_c[y_c==1,0], X_c[y_c==1,1], c='#e74c3c', s=20, alpha=0.7, label='Queimado')
    y_p = rf_cross.predict(X_c)
    f2  = fbeta_score(y_c, y_p, beta=2, zero_division=0)
    ax.set_title(f'{nome}\nF2 = {f2:.4f}')
    ax.set_xlabel('dNBR (simulado)')
    ax.set_ylabel('NBR pós (simulado)')
    if ax == axes[0]: ax.legend(fontsize=8)

plt.suptitle('Generalização Cross-Cena — Modelo treinado na Cena A', fontsize=12)
plt.tight_layout()
plt.show()
print('→ Quanto mais diferente a distribuição da cena de teste, maior a queda de desempenho.')
print('→ Isso motiva treinar com múltiplas cenas para robustez.')

---
## 12. OOB Estimation — Validação Gratuita do Bootstrap

### 12.1 Como funciona

O bootstrap seleciona amostras **com reposição**. A probabilidade de um exemplo específico **não ser selecionado** em uma amostra de tamanho $N$ é:

$$P(\text{não selecionado}) = \left(1 - \frac{1}{N}\right)^N \xrightarrow{N\to\infty} e^{-1} \approx 0.368$$

Portanto, ~36.8% dos exemplos ficam fora de cada árvore. Para cada exemplo $x_i$, calculamos a predição usando **apenas as árvores que não foram treinadas com ele** — isso é o erro OOB.

### 12.2 Por que usar

Com imagens de satélite muito grandes (60M pixels), separar um conjunto de validação desperdiça dados valiosos. O OOB fornece uma estimativa de generalização **sem custo adicional**, usando os próprios exemplos out-of-bag como validação implícita.

In [ ]:
# Demonstra a convergência do erro OOB com o número de árvores
np.random.seed(42)
X_oob, y_oob = make_classification(n_samples=3000, n_features=10, n_informative=5,
                                     n_classes=2, weights=[0.94, 0.06], random_state=42)
X_oob_tr, X_oob_te, y_oob_tr, y_oob_te = train_test_split(
    X_oob, y_oob, test_size=0.3, stratify=y_oob, random_state=42
)

n_trees_range = list(range(1, 201, 5))
oob_scores, test_scores = [], []

for n in n_trees_range:
    rf_oob = RandomForestClassifier(n_estimators=n, oob_score=True,
                                      class_weight='balanced', random_state=42)
    rf_oob.fit(X_oob_tr, y_oob_tr)
    oob_scores.append(rf_oob.oob_score_)
    test_scores.append(accuracy_score(y_oob_te, rf_oob.predict(X_oob_te)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(n_trees_range, oob_scores,  'b-', lw=2, label='OOB Score')
axes[0].plot(n_trees_range, test_scores, 'r--', lw=2, label='Acurácia no Teste')
axes[0].set_xlabel('Número de Árvores (L)')
axes[0].set_ylabel('Acurácia')
axes[0].set_title('OOB Score vs Acurácia no Teste')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Diferença OOB vs Teste
diffs = np.abs(np.array(oob_scores) - np.array(test_scores))
axes[1].plot(n_trees_range, diffs, 'g-', lw=2)
axes[1].fill_between(n_trees_range, 0, diffs, alpha=0.3, color='green')
axes[1].set_xlabel('Número de Árvores (L)')
axes[1].set_ylabel('|OOB - Teste|')
axes[1].set_title('Diferença entre OOB e Teste Real\n(quanto menor, mais confiável o OOB)')
axes[1].grid(alpha=0.3)

plt.suptitle('OOB Estimation — Validação "Gratuita" do Bootstrap', fontsize=12)
plt.tight_layout()
plt.show()

rf_final_oob = RandomForestClassifier(n_estimators=200, oob_score=True,
                                        class_weight='balanced', random_state=42)
rf_final_oob.fit(X_oob_tr, y_oob_tr)
print(f'OOB Score final    : {rf_final_oob.oob_score_:.4f}')
print(f'Acurácia no Teste  : {accuracy_score(y_oob_te, rf_final_oob.predict(X_oob_te)):.4f}')
print(f'Diferença          : {abs(rf_final_oob.oob_score_ - accuracy_score(y_oob_te, rf_final_oob.predict(X_oob_te))):.4f}')
print('→ OOB é um estimador não tendencioso do erro de generalização.')

---
## 13. Resumo: Mapa de Decisões Técnicas

```
PROBLEMA: Classificação binária queimado/não queimado
          com imagens Landsat (pixel = 30m × 30m)
│
├─ DESAFIO 1: NoData (~33% dos pixels)
│   └─ SOLUÇÃO: Máscara booleana antes de qualquer processamento
│               Nunca imputar valores espectrais de satélite
│
├─ DESAFIO 2: Desbalanceamento ~19:1
│   ├─ class_weight='balanced' → penaliza erros na classe queimada
│   ├─ SMOTE → enriquece fronteira de decisão com exemplos sintéticos
│   ├─ Undersampling → reduz custo computacional
│   ├─ SMOTETomek → combina over + limpeza da fronteira
│   └─ Threshold ótimo via curva Precision-Recall (F2)
│
├─ DESAFIO 3: Autocorrelação espacial
│   └─ Blocos espaciais (grid 4×4) → treino/teste em regiões distintas
│      Nunca divisão aleatória de pixels vizinhos
│
├─ MODELO: Random Forest
│   ├─ Base learner CART (entropia → limiares espectrais automáticos)
│   ├─ Bootstrap (diversidade via reamostagem)
│   ├─ Atributos aleatórios √12 ≈ 3 por nó (decorrelação das árvores)
│   ├─ OOB estimation (validação gratuita)
│   └─ Importância de atributos (interpretabilidade física)
│
├─ FEATURES: 12 features por pixel
│   ├─ Pré-evento: B4, B5, B7, NDVI, NBR
│   ├─ Pós-evento: B4, B5, B7, NDVI, NBR
│   └─ Diferenças: dNBR, dNDVI (principais discriminadores)
│
├─ PARAMETRIZAÇÃO: Randomized Search
│   ├─ 30 iterações × 5-fold cross-validation estratificado
│   └─ Scoring: F1 (proxy computacionalmente barato para F2)
│
├─ AVALIAÇÃO: métricas robustas ao desbalanceamento
│   ├─ F2-Score → métrica principal (peso 4× no recall)
│   ├─ MCC → leva em conta todos os quadrantes
│   ├─ Kappa → padrão em sensoriamento remoto
│   └─ AUPRC → curva completa, mais informativa que ROC-AUC
│
└─ GENERALIZAÇÃO: Validação cross-cena
    └─ Treina 221_074 → testa 220_075 (sensor, região e data distintos)
```

---
*Disciplina de Reconhecimento de Padrões — UNESP ICT, Prof. Dr. Rogério Galante Negri.*